<a href="https://colab.research.google.com/github/saifmukadam10/policy-Improvment---RL/blob/main/Policy_Improvement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

In [3]:
transitions = {
    # State 1 transitions
    (1,2,2): 0.8,(1,2,1):0.1,(1,2,-1):0.1, (1,4,-1): 0.8,(1,4,2):0.1,(1,4,-2):0.1,
    # State 2 transitions
    (2,3,2): 0.8,(2,3,-1):0.1,(2,3,1):0.1, (2,1,-2): 0.8,(2,1,1):0.1,(2,1,-1):0.1,
    # State 3 transitions (kept commented as they are unclear/incomplete)
    # (3,2,-2): 0.8,(3,2,1):0.1,(3,2,-1):0.1, (3,6,-1): 0.8,(3,6,2):0.1,(3,6,-2):0.1,
    # (3,2,-2): 0,(3,2,1):0,(3,2,-1):0, (3,6,-1): 0,(3,6,2):0,(3,6,-2):0,
    # State 4 transitions
    (4,1,1): 0.8,(4,1,-2):0.1,(4,1,2):0.1, (4,5,2): 0.8,(4,5,1):0.1,(4,5,-1):0.1,
    # State 5 transitions
    (5,2,1): 0.8,(5,2,-2):0.1,(5,2,2):0.1, (5,4,-2): 0.8,(5,4,1):0.1,(5,4,-1):0.1,
    # State 6 transitions
    (6,3,1): 0.8,(6,3,-2):0.1,(6,3,2):0.1, (6,5,-2): 0.8,(6,5,1):0.1,(6,5,-1):0.1,
    # State 7 transitions
    (7,4,1): 0.8,(7,4,2):0.1,(7,4,-2):0.1, (7,8,2): 0.8,(7,8,1):0.1,(7,8,-1):0.1,
    # State 8 transitions
    (8,9,2): 0.8,(8,9,-1):0.1,(8,9,1):0.1, (8,7,-2): 0.8,(8,7,1):0.1,(8,7,-1):0.1,
    # State 9 transitions
    (9,6,1): 0.8,(9,6,2):0.1,(9,6,-2):0.1, (9,8,-2): 0.8,(9,8,1):0.1,(9,8,-1):0.1
}

In [4]:
transitions

{(1, 2, 2): 0.8,
 (1, 2, 1): 0.1,
 (1, 2, -1): 0.1,
 (1, 4, -1): 0.8,
 (1, 4, 2): 0.1,
 (1, 4, -2): 0.1,
 (2, 3, 2): 0.8,
 (2, 3, -1): 0.1,
 (2, 3, 1): 0.1,
 (2, 1, -2): 0.8,
 (2, 1, 1): 0.1,
 (2, 1, -1): 0.1,
 (4, 1, 1): 0.8,
 (4, 1, -2): 0.1,
 (4, 1, 2): 0.1,
 (4, 5, 2): 0.8,
 (4, 5, 1): 0.1,
 (4, 5, -1): 0.1,
 (5, 2, 1): 0.8,
 (5, 2, -2): 0.1,
 (5, 2, 2): 0.1,
 (5, 4, -2): 0.8,
 (5, 4, 1): 0.1,
 (5, 4, -1): 0.1,
 (6, 3, 1): 0.8,
 (6, 3, -2): 0.1,
 (6, 3, 2): 0.1,
 (6, 5, -2): 0.8,
 (6, 5, 1): 0.1,
 (6, 5, -1): 0.1,
 (7, 4, 1): 0.8,
 (7, 4, 2): 0.1,
 (7, 4, -2): 0.1,
 (7, 8, 2): 0.8,
 (7, 8, 1): 0.1,
 (7, 8, -1): 0.1,
 (8, 9, 2): 0.8,
 (8, 9, -1): 0.1,
 (8, 9, 1): 0.1,
 (8, 7, -2): 0.8,
 (8, 7, 1): 0.1,
 (8, 7, -1): 0.1,
 (9, 6, 1): 0.8,
 (9, 6, 2): 0.1,
 (9, 6, -2): 0.1,
 (9, 8, -2): 0.8,
 (9, 8, 1): 0.1,
 (9, 8, -1): 0.1}

In [7]:
#Q-Values = expected future utility from a q-state (chance node) In [4]:
class MDP:
    def __init__(self, states, action, transition, start,policy, discountf=.99):
        self.states = states
        self.action = action
        self.transition = transition
        self.start = start
        self.discountf = discountf
        self.policy = policy
    def R(self, s):
      #return reward for this state.
      return self.states[s]
    def T(self, s, s_dash,a):
      #for a state and an action, return probability of s'.
      return self.transition[(s,s_dash,a)]
def getpath(newp,start):
    currentstate = start
    itr = 0
    while ((newp[currentstate-1] != 0) and (itr <=9)):
        itr+=1
        x = newp[currentstate-1]
        if (x == 2):
            print("Right")
            if (currentstate % 3 != 0):
                currentstate+=1
        elif (x == -2):
            print("Left")
            if (currentstate % 3 != 1):
                currentstate-=1
        elif (x == -1):
            print("Down")
            if (currentstate <= 6):
                currentstate+=3
        elif (x == 1):
            print("Up")
            if (currentstate >= 4):
                currentstate-=3
    return

In [8]:
def ValueIteration(numofIteration, mdp):
    rewards = mdp.states
    actions = mdp.action

    vPrevois = [[0,0], [0,0], [0,0],
                [0,0], [0,0], [0,0],
                [0,0], [0,0], [0,0]]
    vCurrent = [[0,0], [0,0], [0,0],
                [0,0], [0,0], [0,0],
                [0,0], [0,0], [0,0]]

    for i in range(numofIteration):
        print("itration ", i)
        for s in range(9):
            delta = 0
            Q = [0, 0, 0, 0]
            for a in range(4):
                for s2 in range(9):
                    if (s+1,s2+1,actions[a]) in mdp.transition:
                        #print("s s' a",s+1,s2+1,actions[a], " T ", mdp.transition[(s+1,s2+1
                        Q[a] = Q[a] + mdp.transition[(s+1,s2+1,actions[a])] * (rewards[s]) # Assuming rewards[s] should be used here and not rewards[s2]

            max_index = Q.index(max(Q))
            #print("Q ", Q, " max Q ", max(Q)," max_index ",max_index)
            #print("node ", s+1)
            if s == 0 or s == 2:
                vCurrent[s] = [max(Q),0]
            else:
                vCurrent[s] = [max(Q),actions[max_index]]

            delta = max(delta, abs(vCurrent[s][0] - vPrevois[s][0]))
            vPrevois[s] = vCurrent[s]

        #check for convergence, if values converged then return V
        # Correcting the calculation of the convergence threshold and variable name
        convergence_threshold = (1 - mdp.discountf) / (2 * mdp.discountf)
        print("convergence ", convergence_threshold, " delta ", delta)
        if delta < convergence_threshold and delta != 0:
            print("Early Stop at itration ", i)
            return vCurrent

    return vCurrent